[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Computer Vision - Conditional Diffusion Models

This notebooks applies an image translation model (_Pix2Pix_) using a a Conditional Diffusion generative model.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 08/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0131DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing and Computer Vision

# Machine Learning
from sklearn.model_selection import train_test_split

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import torchinfo

from torchmetrics.functional.image import structural_similarity_index_measure
from torchmetrics.functional.regression import r2_score

import torchvision
from torchvision.io import decode_image
from torchvision.transforms import v2 as TorchVisionTrns

# Miscellaneous
import os
import random
import time
from zipfile import ZipFile

# Typing
from typing import Callable, List, Literal, Optional, Tuple
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)
torch.manual_seed(seedNum) #<! Model init, data order and training noise; `cudnn.benchmark` still adds small run to run differences

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF = (8, 8)

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataManipulation import DownloadUrl
from DeepLearningBlocks import NNMode

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """Converts a PyTorch Tensor to a NumPy array."""
    return tZ.squeeze().detach().cpu().numpy()

def TensorImgNumpy( tI: Tensor ) -> NDArray:
    """Converts a CHW image tensor to an HWC NumPy array."""
    return TensorImageNumpy(tI.permute(1, 2, 0))

class SatAerialMapDataset(Dataset):
    def __init__( self, rootFolderPath: str, dataSet: Literal['Train', 'Validation', 'All'], /, *, imgSize: Optional[int] = None, hTrns: Optional[Callable] = None, geoAug: bool = False ) -> None:
        """
        Satellite Aerial Map Segmentation Dataset.
        The dataset folder structure:
         - Train
            - 00001.jpg
            - 00002.jpg
            - ...
         - Validation
            - 00001.jpg
            - 00002.jpg
            - ...
        Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

        Parameters
        ----------
        rootFolderPath : str
            Path to the folder containing images sets.
        dataSet : Literal['Train', 'Validation', 'All']
            Dataset type to be used. Can be 'Train', 'Validation', or 'All'.
        hTrns : Optional[Callable], optional
            Transform to be applied on the features image (Aerial).
            Should be limited to pixel wise transforms (e.g., normalization, color jitter, etc...).
            By default None.
        geoAug : bool, optional
            Apply a random flip and a random rotation by a multiple of 90 degrees.
            The same transform is applied to both the aerial image and the map, keeping the pair aligned.
            By default False.
        """
        super().__init__()

        if dataSet not in ('Train', 'Validation', 'All'):
            raise ValueError("dataSet must be 'Train', 'Validation', or 'All'")

        lDataSets = ['Train', 'Validation'] if dataSet == 'All' else [dataSet]
        lImgFiles = []
        for dataSetName in lDataSets:
            dataSetFolderPath = os.path.join(rootFolderPath, dataSetName)
            if not os.path.isdir(dataSetFolderPath):
                raise FileNotFoundError(f'Dataset folder does not exist: {dataSetFolderPath}')

            lDataSetFiles = sorted(
                os.path.join(dataSetFolderPath, fileName)
                for fileName in os.listdir(dataSetFolderPath)
                if os.path.isfile(os.path.join(dataSetFolderPath, fileName)) and fileName.lower().endswith(('.jpg', '.jpeg', '.png'))
            )
            lImgFiles.extend(lDataSetFiles)

        self._lImgFiles = lImgFiles
        self._imgSize   = imgSize
        self._hTrns     = hTrns
        self._geoAug    = geoAug

    def __len__( self ) -> int:
        """
        Returns the number of paired images in the dataset.
        """

        return len(self._lImgFiles)

    def __getitem__( self, idx: int ) -> Tuple[Tensor, Tensor]:
        """
        Returns the aerial image and its corresponding map image.

        Parameters
        ----------
        idx : int
            Index of the sample to be fetched.

        Returns
        -------
        Tuple[Tensor, Tensor]
            A tuple containing the aerial image and the map image.
        """
        tPair = decode_image(self._lImgFiles[idx], mode = 'RGB')
        imgWidth = tPair.shape[2]

        imgWidthHalf = imgWidth // 2
        tX = tPair[:, :, :imgWidthHalf]
        tY = tPair[:, :, imgWidthHalf:]

        if self._imgSize is not None:
            tX = TorchVisionTrns.functional.resize(tX, size = (self._imgSize, self._imgSize), interpolation = TorchVisionTrns.InterpolationMode.BILINEAR, antialias = True)
            tY = TorchVisionTrns.functional.resize(tY, size = (self._imgSize, self._imgSize), interpolation = TorchVisionTrns.InterpolationMode.BILINEAR, antialias = True)

        if self._geoAug:
            # Same flip / rotation on both images keeps the pair aligned (covers all 8 symmetries of the square)
            if random.random() < 0.5:
                tX = tX.flip(-1)
                tY = tY.flip(-1)
            numRot = random.randrange(4)
            tX = torch.rot90(tX, numRot, dims = (-2, -1))
            tY = torch.rot90(tY, numRot, dims = (-2, -1))

        if self._hTrns:
            tX = self._hTrns(tX)

        tY = TorchVisionTrns.functional.to_dtype(tY, torch.float, scale = True)

        return tX, tY

    def SetImageSize( self, imgSize: Optional[int] ) -> None:
        """
        Sets the image size for resizing the aerial and map images.

        Parameters
        ----------
        imgSize : Optional[int]
            The desired image size. If None, no resizing will be applied.
        """
        self._imgSize = imgSize

    def SetTransforms( self, hTrns: Optional[Callable] ) -> None:
        """
        Sets the pixel wise transforms applied to the aerial image.
        """
        self._hTrns = hTrns

* <font color='blue'>(**!**)</font> Inspect `SatAerialMapDataset`: each paired image contains the aerial image on the left and its aligned RGB map on the right.
* <font color='brown'>(**#**)</font> Maps are RGB regression targets, not integer segmentation labels. Noise targets are generated later in the training loop.

## Conditional Diffusion Model

A _Conditional Diffusion Model_ uses extra information at each denoising step.  
This _Condition_ can be a class label, text or an image.

The goal of those 2 models is different:
 - Diffusion Model: Generate a plausible sample.
 - Conditional Diffusion Model: Generate a plausible sample that matches the conditional information.

In this notebook, the condition is an aerial image. The generated sample is its corresponding map.

- **Training:** Add noise to the target map. Recover it (predict the noise, or the clean map) using the noisy map, the timestep, and the aerial image.
- **Sampling:** Start the map from random noise. Keep the aerial image fixed throughout denoising.
- **Key Distinction:** Only the target map is diffused. The aerial image provides spatial information about roads, buildings, and other features.

The denoising objective stays the same. The denoiser gains an extra input: the condition.

Next, _Classifier Free Guidance (CFG)_ lets us adjust the influence of that condition during sampling, without retraining.


### Denoising Diffusion Probabilistic Model (DDPM) Target

**One forward process, three equivalent regression targets.**

The forward process mixes the clean target $\boldsymbol{y}_0$ with Gaussian noise:

$$ \boldsymbol{y}_t = \sqrt{\bar\alpha_t} \boldsymbol{y}_0 + \sqrt{1 - \bar\alpha_t} \boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon} \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right), \qquad \text{SNR}_t = \frac{\bar\alpha_t}{1 - \bar\alpha_t}. $$

Given $\boldsymbol{y}_t$ and $t$, knowing any one of $\boldsymbol{y}_0$, $\boldsymbol{\epsilon}$ or their mixture $\boldsymbol{v}$ determines the other two. The network may be trained to output any of them; the sampler always converts the output to a clean estimate $\hat{\boldsymbol{y}}_0$ (`PredictClean()`) and proceeds identically. The three variants share the model, the schedule, the sampler and CFG. They differ in what the last layer emits and in how the MSE weights the noise levels.

| Variant (`predictType`) | Network output | Clean estimate $\hat{\boldsymbol{y}}_0$ | Implied weight on ${\left\| \boldsymbol{y}_0 - \hat{\boldsymbol{y}}_0 \right\|}^{2}$ |
|---|---|---|---|
| Noise (`'Noise'`) | $\hat{\boldsymbol{\epsilon}}$ | $\left( \boldsymbol{y}_t - \sqrt{1 - \bar\alpha_t} \hat{\boldsymbol{\epsilon}} \right) / \sqrt{\bar\alpha_t}$ | $\text{SNR}_t$ |
| Clean (`'Clean'`) | $\hat{\boldsymbol{y}}_0$ | $\hat{\boldsymbol{y}}_0$ | $1$ |
| Velocity | $\hat{\boldsymbol{v}}$ | $\sqrt{\bar\alpha_t} \boldsymbol{y}_t - \sqrt{1 - \bar\alpha_t} \hat{\boldsymbol{v}}$ | $1 + \text{SNR}_t$ |

with $\boldsymbol{v} \triangleq \sqrt{\bar\alpha_t} \boldsymbol{\epsilon} - \sqrt{1 - \bar\alpha_t} \boldsymbol{y}_0$.

The last column follows from the forward process with $\boldsymbol{y}_t$ held fixed: ${\left\| \boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}} \right\|}^{2} = \text{SNR}_t {\left\| \boldsymbol{y}_0 - \hat{\boldsymbol{y}}_0 \right\|}^{2}$ and ${\left\| \boldsymbol{v} - \hat{\boldsymbol{v}} \right\|}^{2} = \left( 1 + \text{SNR}_t \right) {\left\| \boldsymbol{y}_0 - \hat{\boldsymbol{y}}_0 \right\|}^{2}$. Since the timestep is drawn uniformly, *MSE on the chosen target* is *MSE on the clean map weighted by $w \left( t \right)$*. With the cosine schedule, $\text{SNR}_t$ spans $\approx 10^{4}$ (almost clean) to $\approx 10^{-4}$ (almost pure noise).

#### Where "Velocity" Comes From

Write $\sqrt{\bar\alpha_t} = \cos \phi_t$ and $\sqrt{1 - \bar\alpha_t} = \sin \phi_t$. Then $\boldsymbol{y}_t = \cos \phi_t \boldsymbol{y}_0 + \sin \phi_t \boldsymbol{\epsilon}$ moves on a circle from the data ($\phi = 0$) to pure noise ($\phi = \pi / 2$), and

$$ \boldsymbol{v} = \frac{\mathrm{d} \boldsymbol{y}_t}{\mathrm{d} \phi} = -\sin \phi_t \boldsymbol{y}_0 + \cos \phi_t \boldsymbol{\epsilon} $$

is the tangent, the *velocity* along the arc. It has unit variance at every $\phi$ and is never degenerate: at $\phi = 0$ it equals $\boldsymbol{\epsilon}$, at $\phi = \pi / 2$ it equals $-\boldsymbol{y}_0$. It rotates smoothly from "predict the noise" to "predict the image".

#### Why the Choice Matters

A perfectly trained network is identical under all three. A finite network trained for a finite budget is not:

**Conditioning of the output.**
 - Noise at high noise: $\hat{\boldsymbol{\epsilon}} \approx \boldsymbol{y}_t / \sqrt{1 - \bar\alpha_t}$. The network must reproduce its input and hide all information about the map in a residual scaled by $\sqrt{\bar\alpha_t} \approx 0.01$; the sampler then divides by $\sqrt{\bar\alpha_t}$ and amplifies any output error $\approx 100 \times$. Ill conditioned exactly where the condition matters most.
 - Clean at low noise: the network must reproduce $\boldsymbol{y}_t \approx \boldsymbol{y}_0$. Trivial, yet the small weight means fine detail is barely refined there.
 - Velocity: unit variance target at every $t$, no division anywhere.

**What gets learned first.** The weight $w \left( t \right)$ is a curriculum:
 - $\text{SNR}_t$: almost all of the gradient goes to low noise (fine detail). The high noise regime, the only place a conditional model *must* read its condition, receives $\approx 10^{-4}$ of the signal. In this notebook, after 10 epochs of noise prediction a *wrong* aerial image changed the loss by less than 4%: the network had learned an unconditional map denoiser.
 - $1$: uniform. The high noise samples (aerial $\to$ map regression, the hardest) dominate; conditioning is learned from the first iterations. Low noise refinement is under trained.
 - $1 + \text{SNR}_t$: $\approx 1$ at high noise (keeps the conditioning), $\approx \text{SNR}_t$ at low noise (restores edge refinement). The usual compromise (Imagen Video, Stable Diffusion 2.x).

**Sampling.** Only the conversion to $\hat{\boldsymbol{y}}_0$ differs. CFG is linear, so it applies to any of the three outputs.

#### Which to Pick

 - Unconditional natural images, long training (DDPM, ADM): noise. High noise carries little to learn; perceptual quality lives at low noise.
 - Strongly conditional, near deterministic mapping, limited budget (this notebook): clean. Learn the regressor first; accept softer edges.
 - Both, or large $T$ / high resolution where $\bar\alpha_T \to 0$ makes the noise target degenerate: velocity.

* <font color='brown'>(**#**)</font> All three are cases of $\hat{\boldsymbol{y}}_0 = c_{\text{skip}} \left( t \right) \boldsymbol{y}_t + c_{\text{out}} \left( t \right) F_{\theta} \left( \cdot \right)$ with different coefficients, plus a loss weight $w \left( t \right)$ (Karras et al., EDM 2022). Loss re weighting schemes (P2, Min-SNR-$\gamma$) turn the second knob while keeping the noise output: they can match the curriculum of the other variants yet not their conditioning.
* <font color='brown'>(**#**)</font> Velocity prediction is not implemented in this notebook. It requires 3 lines: `Target()` returns $\sqrt{\bar\alpha_t} \boldsymbol{\epsilon} - \sqrt{1 - \bar\alpha_t} \boldsymbol{y}_0$, `PredictClean()` returns $\sqrt{\bar\alpha_t} \boldsymbol{y}_t - \sqrt{1 - \bar\alpha_t} \hat{\boldsymbol{v}}$, and `predictType` accepts `'Velocity'`.
* <font color='red'>(**?**)</font> The training loss values of two prediction types are not comparable. Why? What *is* comparable?
* <font color='green'>(**@**)</font> Implement velocity prediction and compare the generated maps with clean prediction at the same epoch.

### Conditional Diffusion with Classifier Free Guidance

#### Motivation

An unconditional diffusion model can generate a plausible map, but it does not know which roads, buildings, and parks belong to a particular aerial image. A conditional model receives that image and learns to use it. At sampling time, we may want stronger adherence to the source than ordinary conditional sampling provides.

Classifier Free Guidance (CFG) provides a sampling time control over that adherence, without training a separate classifier.  
Here the condition is an image, not a class label; the same idea applies to text and other conditions.

#### Training One Model with Two Tasks

The source aerial image $\boldsymbol{x}$ conditions generation of its aligned target map $\boldsymbol{y}_0$. Only the target is diffused:

$$ \boldsymbol{y}_t = \sqrt{\bar\alpha_t}\boldsymbol{y}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon}\sim\mathcal N(\boldsymbol{0},\boldsymbol{I}). $$

Here $\alpha_t=1-\beta_t$ and $\bar\alpha_t=\prod_{s=1}^t\alpha_s$. Source and target are scaled from $[0,1]$ to $[-1,1]$ outside the dataset. The aerial image is not diffused.

During training, independently drop the entire aerial condition for each example with probability `conditionDropProb`. Dropping means setting the source channels to zero and the presence mask to zero. Otherwise, the original source is supplied with a mask of one. This distinguishes a missing condition from a real image whose normalized pixels happen to be zero.

The same U-Net learns conditional and unconditional noise prediction with the same noise target and loss:

$$ \mathcal L(\theta)=\mathbb E\left[\left\|\boldsymbol{\epsilon}-\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\boldsymbol{c})\right\|_2^2\right], \qquad \boldsymbol{c}\in\{\boldsymbol{x},\varnothing\}. $$

```mermaid
flowchart LR
    Clean[Clean target map] --> Forward[Add sampled noise at time t]
    Noise[Sampled noise] --> Forward
    Source[Aerial image] --> Drop{Drop condition?}
    Drop -->|No| Present[Source image and mask one]
    Drop -->|Yes| Null[Zero image and mask zero]
    Forward --> Net[Shared time-conditioned U-Net]
    Present --> Net
    Null --> Net
    Net --> Loss[Noise prediction MSE]
    Noise --> Loss
```

The null condition task learns what maps generally look like. The conditional task additionally learns how the source changes the denoising prediction. Condition dropout is not dropout of hidden network activations.

#### Intuition at Sampling Time

At the same noisy map and timestep, ask the shared network twice:

- Without the source: what noise should be removed to move toward a plausible map?
- With the source: what noise should be removed to move toward a map compatible with this aerial image?

The difference between these noise predictions captures the effect of supplying the source. CFG amplifies that difference:

$$ \hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\varnothing) + w\left[\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\boldsymbol{x})-\boldsymbol{\epsilon}_\theta(\boldsymbol{y}_t,t,\varnothing)\right]. $$

For $w>1$, this extrapolates beyond the conditional prediction; it is not an average of two generated images. Under the usual noise-to-score conversion, the same combination amplifies the condition-dependent contribution to the score.

```mermaid
flowchart LR
    State[Current noisy map and time] --> Uncond[Shared U-Net with null condition]
    State --> Cond[Shared U-Net with aerial condition]
    Source[Fixed aerial image] --> Cond
    Uncond --> CFG[Combine noise predictions using guidance scale]
    Cond --> CFG
    CFG --> Step[One DDPM reverse step]
    Step --> Next[Less noisy map]
```

- $w=0$: unconditional map generation.
- $w=1$: ordinary conditional DDPM.
- $w>1$: amplify the effect of the aerial condition; excessive guidance can cause artifacts or reduce diversity.

Guidance changes sampling, not the trained weights. Stronger guidance is not guaranteed to improve pixel accuracy, and it does not enforce exact road alignment. The comparison at the end of the notebook keeps both initial and per-step noise fixed so differences reflect the guidance scale.

The guided prediction determines the DDPM reverse mean. The schedule adds fresh Gaussian noise at each reverse step except the last. Every step uses the same network weights; no classifier is trained.

* <font color='brown'>(**#**)</font> This is paired image translation, not the original adversarial Pix2Pix method. It uses no discriminator.
* <font color='brown'>(**#**)</font> Source and target must remain spatially aligned. Do not apply an independent geometric augmentation to only one member of the pair.
* <font color='brown'>(**#**)</font> Reference: [Classifier Free Diffusion Guidance](https://arxiv.org/abs/2207.12598).

In [ ]:
# Parameters

# Data
dataSet    = 'SatAerialToMap'
dataSetUrl = r'https://huggingface.co/datasets/Royi/DataSets/resolve/main/SatAerialToMap.zip'
imgSize = 256
trainNumSamples = None
valNumSamples = 32

# Model
baseCh = 32
lChMult = (2, 3, 6, 8) #<! Channels per level: 64 / 96 / 192 / 256 (must match the trained checkpoint)
numBlocks = 2
useSeparable = False
numDiffSteps = 200
predictType = 'Clean' #<! 'Noise' or 'Clean'
conditionDropProb = 0.1
guidanceScale = 2.0

# Training
batchSize = 8
numWorkers = 0
numEpochs = 200
scoreType = 'R2'

# Validation
valEvery = 10

# Optimizer
ηOpt = 2e-4 #<! Peak learning rate
tuβ = (0.9, 0.99)
weightDecay = 5e-5
ηMin = 1e-5 #<! Floor of the cosine decay
warmupFrac = 0.05 #<! Linear warmup, fraction of the epochs
holdFrac = 0.45 #<! Hold at the peak, fraction of the epochs (the rest is cosine decay)

# Visualization
numImg = 3
numPlotSteps = 6

## Generate / Load Data

Use the [SatAerialToMap dataset](https://huggingface.co/datasets/Royi/DataSets). Each file contains an aerial/map pair. Reuse existing local files, or download and extract the archive.

We concatenate the supplied `Train` and `Validation` folders and use a seeded split with `valNumSamples = 32`. With `trainNumSamples = None`, all remaining pairs are training samples. Separate dataset instances allow paired geometric and source-only photometric augmentation during training and deterministic validation.

* <font color='brown'>(**#**)</font> This uses a new split, not the archive's original holdout. Nearby geographic tiles may be correlated; geographic splits are preferable for measuring generalization to new locations.


In [ ]:
# Extract Files
# Will create:
# - `FixelCourses/DataSets/SatAerialToMap/Train` - Contains all images.
# - `FixelCourses/DataSets/SatAerialToMap/Validation` - Contains all images.
# Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

datasetFolderPath     = os.path.join(DATA_FOLDER_PATH, dataSet)

# Delete existing folders if any
if not os.path.isdir(datasetFolderPath):
    # 1. Download the ZIP file by URL.
    # 2. Extract the ZIP file to the dataset folder path.
    # 3. Delete the ZIP file.

    fileName = os.path.join(DATA_FOLDER_PATH, f'{dataSet}.zip')
    DownloadUrl(dataSetUrl, DATA_FOLDER_PATH)
    with ZipFile(fileName, 'r') as zipFile:
        zipFile.extractall(DATA_FOLDER_PATH) #<! The Zip file contains a folder
    time.sleep(1.0) #<! Wait for the file system to update
    os.remove(fileName)

In [ ]:
# Data Set

oTrnsDisplay = TorchVisionTrns.ToDtype(torch.float32, scale = True)
dsData = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsDisplay)
numSamples = len(dsData)
print(f'Number of paired aerial/map images: {numSamples}')

In [ ]:
# Element of the Data Set

tX, tY = dsData[0]
print(f'Aerial image: {tX.shape}, {tX.dtype}, range [{tX.min():.2f}, {tX.max():.2f}]')
print(f'Target map  : {tY.shape}, {tY.dtype}, range [{tY.min():.2f}, {tY.max():.2f}]')

### Plot the Data

In [ ]:
# Plot Paired Data

hF, mHa = plt.subplots(numImg, 2, figsize = (8, 4 * numImg), squeeze = False)
for sampleIdx in range(numImg):
    tX, tY = dsData[random.randrange(numSamples)]
    for hA, tImage, title in zip(mHa[sampleIdx], (tX, tY), ('Aerial Image', 'Target Map')):
        hA.imshow(TensorImgNumpy(tImage))
        hA.set_title(title)
        hA.axis('off')
hF.tight_layout();

### Augmentation / Transform

Two kinds of augmentation are applied during training only:

1. **Geometric (paired)** - A random horizontal flip and a random rotation by a multiple of $90^{\circ}$, applied identically to the aerial image and the map (`geoAug = True` in the dataset). The pair stays aligned. Together the flip and rotation cover all 8 symmetries of the square. Maps have no preferred orientation, so this multiplies the effective data set size by 8 without changing the task.
2. **Photometric (source only)** - Applied only to the aerial image. The target map is unchanged and remains in $[0,1]$ at the dataset output.

The training and sampling functions convert image values to $[-1,1]$. Diffusion noise is generated in the training loop, independently of source augmentation.


In [ ]:
# Source Image Transforms

oTrnsTrain = TorchVisionTrns.Compose([
    TorchVisionTrns.ToDtype(torch.float32, scale = True),
    TorchVisionTrns.RandomChoice([
        TorchVisionTrns.RandomGrayscale(p = 1.0),
        TorchVisionTrns.GaussianBlur(7, sigma = (0.1, 1.0)),
        TorchVisionTrns.RandomEqualize(p = 1.0),
        TorchVisionTrns.RandomAutocontrast(p = 1.0),
        TorchVisionTrns.GaussianNoise(sigma = 0.05),
        TorchVisionTrns.RandomErasing(p = 1.0, scale = (0.05, 0.15), ratio = (0.5, 2.0), value = 0, inplace = True),
        TorchVisionTrns.RGB(),
    ], p = [0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.58]),
])
oTrnsVal = TorchVisionTrns.ToDtype(torch.float32, scale = True)

In [ ]:
# Create Training and Validation Datasets

dsTrain = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsTrain, geoAug = True)
dsVal   = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsVal)

vIdxTrain, vIdxVal = train_test_split(np.arange(numSamples), test_size = valNumSamples, train_size = trainNumSamples, random_state = seedNum, shuffle = True)

dsTrain = torch.utils.data.Subset(dsTrain, vIdxTrain)
dsVal = torch.utils.data.Subset(dsVal, vIdxVal)

print(f'The training data set contains  : {len(dsTrain):4d} samples.')
print(f'The validation data set contains: {len(dsVal):4d} samples.')

In [ ]:
# Inspect an Augmented Pair

tX, tY = dsTrain[0]
print(f'Source: {tX.shape}, target: {tY.shape}')
hF, vHa = plt.subplots(1, 2, figsize = (8, 4))
for hA, tImage, title in zip(vHa, (tX, tY), ('Augmented Aerial', 'Target Map (Same Flip / Rotation)')):
    hA.imshow(TensorImgNumpy(tImage).clip(0, 1))
    hA.set_title(title)
    hA.axis('off')
hF.tight_layout();

In [ ]:
# Forward and Reverse Diffusion Schedule

class DiffusionSchedule:
    def __init__( self, numSteps: int, runDevice: torch.device = torch.device('cpu'), *, vMean: Tuple[float, float, float] = (0.5, 0.5, 0.5), dataStd: float = 0.5, predictType: Literal['Noise', 'Clean'] = 'Noise' ) -> None:
        # Assumes numSteps >= 2. Construct on the same device as the images.
        # Diffusion runs on the normalized target (y - vMean) / dataStd so the data is zero mean like the noise prior; defaults reproduce 2y - 1.
        # predictType: what the network outputs, the added noise ('Noise') or the clean map ('Clean'); both give the clean estimate used by `Step`.
        if predictType not in ('Noise', 'Clean'):
            raise ValueError("predictType must be 'Noise' or 'Clean'")
        self.numSteps = numSteps
        self.predictType = predictType
        vGrid = torch.linspace(0, 1, numSteps + 1, dtype = torch.float64)
        vCurve = torch.cos((vGrid + 0.008) / 1.008 * np.pi / 2).square()
        vBeta = (1 - vCurve[1:] / vCurve[:-1]).clamp(1e-5, 0.999)
        vAlpha = 1 - vBeta
        vAlphaBar = torch.cumprod(vAlpha, dim = 0)
        vAlphaPrev = torch.cat((torch.ones(1, dtype = torch.float64), vAlphaBar[:-1]))
        self.vBeta = vBeta.float().to(runDevice)
        self.vAlphaBar = vAlphaBar.float().to(runDevice)
        self.vVariance = (vBeta * (1 - vAlphaPrev) / (1 - vAlphaBar)).float().to(runDevice)
        self.vCoefClean = (vBeta * vAlphaPrev.sqrt() / (1 - vAlphaBar)).float().to(runDevice)
        self.vCoefNoisy = (vAlpha.sqrt() * (1 - vAlphaPrev) / (1 - vAlphaBar)).float().to(runDevice)
        self.tMean = torch.tensor(vMean).view(1, -1, 1, 1).to(runDevice)
        self.dataStd = dataStd
        self.tClampMin = self.Normalize(torch.zeros_like(self.tMean)) #<! Image value 0 in diffusion space
        self.tClampMax = self.Normalize(torch.ones_like(self.tMean)) #<! Image value 1 in diffusion space

    def Normalize( self, tImg: Tensor ) -> Tensor:
        # Image in [0, 1] -> diffusion space
        return (tImg - self.tMean) / self.dataStd

    def Denormalize( self, tClean: Tensor ) -> Tensor:
        # Diffusion space -> image in [0, 1]
        return (tClean * self.dataStd + self.tMean).clamp(0, 1)

    def AddNoise( self, tClean: Tensor, vTime: Tensor, tNoise: Tensor ) -> Tensor:
        # Assumes BCHW images, matching noise, and integer timestep indices in [0, numSteps).
        tAlphaBar = self.vAlphaBar[vTime].view(-1, 1, 1, 1)
        return tAlphaBar.sqrt() * tClean + (1 - tAlphaBar).sqrt() * tNoise

    def Target( self, tClean: Tensor, tNoise: Tensor ) -> Tensor:
        # The regression target of the network
        return tNoise if self.predictType == 'Noise' else tClean

    def PredictClean( self, tNoisy: Tensor, tPred: Tensor, vTime ) -> Tensor:
        # Clean map estimate from the network output (`vTime` is a batch of indices or a single index)
        if self.predictType == 'Clean':
            return tPred
        tAlphaBar = self.vAlphaBar[vTime].view(-1, 1, 1, 1)
        return (tNoisy - (1 - tAlphaBar).sqrt() * tPred) / tAlphaBar.sqrt()

    def Step( self, tNoisy: Tensor, tPred: Tensor, stepIdx: int, tNoise: Tensor ) -> Tensor:
        # Assumes matching BCHW tensors and 0 <= stepIdx < numSteps.
        tClean = self.PredictClean(tNoisy, tPred, stepIdx).clamp(self.tClampMin, self.tClampMax)
        tMean = self.vCoefClean[stepIdx] * tClean + self.vCoefNoisy[stepIdx] * tNoisy
        return tMean + self.vVariance[stepIdx].sqrt() * tNoise

* <font color='brown'>(**#**)</font> Code index `0` is the first noisy state ($t=1$). The clean target is $\boldsymbol{y}_0$. The cosine schedule makes the final state approximately standard Gaussian.
* <font color='brown'>(**#**)</font> **Normalization.** The noise prior is $\mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right)$, so the data must be zero mean and roughly unit variance as well. Maps in $[0, 1]$ have mean $\approx 0.9$ and std $\approx 0.08$: mapping them with $2y - 1$ gives mean $0.8$ and std $0.17$, a poor match. `Normalize()` uses the per channel mean and a single std of the training maps; a single std keeps the color proportions. The schedule owns both directions since the sampler must undo the normalization and clamp to the valid image range.
* <font color='brown'>(**#**)</font> **Prediction type.** The network may output the added noise $\boldsymbol{\epsilon}$ or the clean map $\boldsymbol{y}_0$. Given $\boldsymbol{y}_t$ the two are equivalent ($\boldsymbol{y}_t = \sqrt{\bar\alpha_t} \boldsymbol{y}_0 + \sqrt{1 - \bar\alpha_t} \boldsymbol{\epsilon}$), yet they weight the training differently, see the training section. `Target()` and `PredictClean()` hide the choice from the rest of the code.
* <font color='brown'>(**#**)</font> `Step()` clips only the clean map estimate to the valid image range, then uses it to compute the reverse mean. Reverse variance is $\tilde\beta_t=\beta_t(1-\bar\alpha_{t-1})/(1-\bar\alpha_t)$, which is zero at the final step. Intermediate noisy states are not clipped.
* <font color='red'>(**?**)</font> Why must flips and rotations be applied jointly to the aerial image and map?


### Data Loaders

In [ ]:
# Data Loaders

pinMemory = torch.cuda.is_available()
persWork = numWorkers > 0
prefetchFactor = 2 if persWork else None

dlTrain = torch.utils.data.DataLoader(dsTrain, shuffle = True, batch_size = batchSize, num_workers = numWorkers, pin_memory = pinMemory, drop_last = True, persistent_workers = persWork, prefetch_factor = prefetchFactor)
dlVal = torch.utils.data.DataLoader(dsVal, shuffle = False, batch_size = batchSize, num_workers = numWorkers, pin_memory = pinMemory, drop_last = False, persistent_workers = persWork, prefetch_factor = prefetchFactor)

### Target Map Statistics

One pass over the training loader gives the per channel mean and the global std of the maps. They define the diffusion space of the schedule.

* <font color='brown'>(**#**)</font> Statistics are computed on the training split only. The flips and rotations of the training set do not change them.
* <font color='brown'>(**#**)</font> A single (global) std keeps the RGB proportions of the map colors. Per channel stds ($\approx 0.09, 0.06, 0.12$) would stretch green about twice as much as blue and weight the channels unequally in the loss.


In [ ]:
# Target Map Statistics

def ComputeMapStats( dlData ) -> Tuple[Tuple[float, float, float], float]:
    # One pass over the training loader: per channel mean and a single (global) std of the target maps in [0, 1]
    vSum = torch.zeros(3, dtype = torch.float64)
    sumSq = 0.0
    numPix = 0
    for _, tY in dlData:
        vSum += tY.sum(dim = (0, 2, 3)).double()
        sumSq += tY.double().square().sum().item()
        numPix += tY.shape[0] * tY.shape[2] * tY.shape[3]

    vMean = vSum / numPix
    mapStd = np.sqrt(sumSq / (3 * numPix) - vMean.mean().item() ** 2)

    return tuple(vMean.tolist()), float(mapStd)

vMapMean, mapStd = ComputeMapStats(dlTrain)
print(f'Target map mean (RGB): {vMapMean[0]:.3f}, {vMapMean[1]:.3f}, {vMapMean[2]:.3f}, std: {mapStd:.3f}')

oDiff = DiffusionSchedule(numDiffSteps, vMean = vMapMean, dataStd = mapStd, predictType = predictType) #<! On CPU, for the demonstration below

In [ ]:
# Forward Diffusion of a Target Map

tX, tY = next(iter(dlVal))
print(f'Aerial batch: {tX.shape}, map batch: {tY.shape}')
tClean = oDiff.Normalize(tY[:1]).repeat(numPlotSteps, 1, 1, 1)
vTime = torch.linspace(0, numDiffSteps - 1, numPlotSteps).long()
tNoisy = oDiff.AddNoise(tClean, vTime, torch.randn_like(tClean))
hF, vHa = plt.subplots(1, numPlotSteps + 2, figsize = (3 * (numPlotSteps + 2), 3))
vHa[0].imshow(TensorImgNumpy(tX[0])); vHa[0].set_title('Aerial Condition')
vHa[1].imshow(TensorImgNumpy(tY[0])); vHa[1].set_title('Clean Map')
for plotIdx, hA in enumerate(vHa[2:]):
    hA.imshow(TensorImgNumpy(oDiff.Denormalize(tNoisy[plotIdx:(plotIdx + 1)])))
    hA.set_title(f'Step {vTime[plotIdx].item() + 1}')
for hA in vHa:
    hA.axis('off')
hF.tight_layout();

## Build the Conditional Denoiser

A convolutional U-Net outputs three channels: the added noise or the clean map, according to `predictType`. It receives seven input channels: noisy RGB map, normalized RGB aerial image, and a binary condition-presence mask. When the condition is dropped, the aerial channels and mask are zero.

Four downsampling stages capture context; bilinear resizing and skip connections restore spatial detail. GroupNorm supports small batches and mixed noise levels. A sinusoidal timestep embedding is projected into every residual block.

### Depth per Level

Each encoder / decoder level is a `TimeStage`: a stack of `numBlocks = 2` residual `TimeBlock`s. The first block changes the channel count, the rest keep it.

**Why depth at every level and not only wider channels?**  
Widening (`baseCh`) grows the parameter count quadratically at the bottleneck, where the $16 \times 16$ grid already holds most of the parameters.  
A second block per level adds few parameters at the $256 \times 256$ and $128 \times 128$ levels, yet doubles the number of non-linear layers that operate on the fine structure of the map: thin roads, block boundaries, text like edges. These are the details a pixel-wise score is sensitive to.

### Width per Level

`lChMult` sets the channel count of each level as a multiple of `baseCh`. The classic layout is $(1, 2, 4, 8)$: channels double as the resolution halves.

In this U-Net, with doubling channels, **every level costs about the same FLOPs, yet the parameters sit almost entirely in the deep levels**: a $3 \times 3$ convolution at $256^2$ with $32$ channels has $\approx 9$K parameters; the same convolution at $32^2$ with $256$ channels has $\approx 590$K. Uniform widening (`baseCh = 64`) therefore spends most of its $4 \times$ parameters at the bottleneck, which is not where thin roads and block edges are formed.

The layout used here, $(2, 3, 6, 8)$ = $64 / 96 / 192 / 256$ channels, widens the three finer levels and leaves the $16 \times 16$ bottleneck unchanged. Compared to $(1, 2, 4, 8)$ it adds $\approx 28\%$ parameters ($8.9$M $\to$ $11.4$M) and $\approx 2 \times$ compute, all of it at the resolutions where detail is produced.

* <font color='brown'>(**#**)</font> Two residual blocks per level is the standard DDPM / ADM layout. With `baseCh = 32` and `lChMult = (1, 2, 4, 8)` the model has ~8.9M parameters (~5.6M with a single block).
* <font color='brown'>(**#**)</font> Compute grows roughly with the number of blocks at the high resolution levels, since a $3 \times 3$ convolution at $256 \times 256$ costs more FLOPs than one at $32 \times 32$ with $8$ times the channels.
* <font color='brown'>(**#**)</font> The loss per noise level tells which levels are limiting: low noise steps (fine detail) stress the wide levels, high noise steps (the aerial $\to$ map regression) stress the deep ones.

### Bottleneck Attention

**Convolutions are local. Attention is global.**

A $3 \times 3$ convolution mixes each position with its 8 neighbors only.  
Stacking layers and pooling grows the receptive field, yet distant positions interact only through many intermediate layers.

At the bottleneck the $256 \times 256$ image is a $16 \times 16$ grid. Each position summarizes a $16 \times 16$ pixel patch.  
Self attention lets every bottleneck position look at all $256$ positions in a single layer:

1. Normalize the features. Project each position to a query $\boldsymbol{q}$, a key $\boldsymbol{k}$ and a value $\boldsymbol{v}$ using $1 \times 1$ convolutions.
2. Compare each query with all keys: $\operatorname{softmax} \left( \boldsymbol{Q} \boldsymbol{K}^{T} / \sqrt{d} \right)$. Each row is a set of weights over all positions.
3. Replace each position by the weighted average of all values. Add the result to the input (residual).

The block runs once, at the bottleneck, where it is cheap. Its cost grows with the square of the number of positions:
 - $16 \times 16 = 256$ positions: $256^2 \approx 6.5 \cdot 10^{4}$ similarity scores per head.
 - $256 \times 256$ positions: $\left( 6.5 \cdot 10^{4} \right)^2 \approx 4.3 \cdot 10^{9}$. Not feasible at full resolution.

For maps, global mixing helps:
 - A road entering a tile on the left should continue consistently to the right.
 - Water, parks and residential blocks should keep one color across a region.
 - Aerial context from the whole tile informs each local decision.

* <font color='brown'>(**#**)</font> The output projection is zero initialized. The block starts as the identity and learns how much to rely on attention. Early training matches the purely convolutional model.
* <font color='brown'>(**#**)</font> Multi head attention (`numHeads = 4`) splits the channels into groups, each with its own weights over positions. It adds no parameters compared to a single head.
* <font color='brown'>(**#**)</font> The block has no time input. The timestep enters through the residual blocks before and after it.
* <font color='brown'>(**#**)</font> Attention at the $32 \times 32$ level ($1024$ positions) is also common, at $16$ times the compute of the bottleneck block.


In [ ]:
# Time Conditioned U-Net

class TimeBlock(nn.Module):
    def __init__( self, inCh: int, outCh: int, timeDim: int, *, useSeparable: bool = True ) -> None:
        # Assumes outCh is divisible by 8 for GroupNorm.
        super().__init__()
        oConv = nn.Conv2d(inCh, outCh, 3, padding = 1)
        if useSeparable and min(inCh, outCh) >= 64:
            oConv = nn.Sequential(nn.Conv2d(inCh, inCh, 3, padding = 1, groups = inCh, bias = False), nn.Conv2d(inCh, outCh, 1)) #<! Spatial filtering per channel, then channel mixing
        oOut = nn.Conv2d(outCh, outCh, 3, padding = 1)
        if useSeparable and outCh >= 64:
            oOut = nn.Sequential(nn.Conv2d(outCh, outCh, 3, padding = 1, groups = outCh, bias = False), nn.Conv2d(outCh, outCh, 1))
        self.oConv = nn.Sequential(oConv, nn.GroupNorm(8, outCh), nn.SiLU())
        self.oTime = nn.Linear(timeDim, outCh)
        self.oOut = nn.Sequential(nn.GroupNorm(8, outCh), nn.SiLU(), oOut)
        self.oSkip = nn.Conv2d(inCh, outCh, 1) if inCh != outCh else nn.Identity()

    def forward( self, tX: Tensor, mTime: Tensor ) -> Tensor:
        tZ = self.oConv(tX) + self.oTime(mTime)[:, :, None, None]
        return self.oOut(tZ) + self.oSkip(tX)

class TimeStage(nn.Module):
    def __init__( self, inCh: int, outCh: int, timeDim: int, numBlocks: int, *, useSeparable: bool = True ) -> None:
        # Assumes numBlocks >= 1. The first block changes the channel count, the rest keep it.
        super().__init__()
        self.lBlocks = nn.ModuleList([TimeBlock(inCh if ii == 0 else outCh, outCh, timeDim, useSeparable = useSeparable) for ii in range(numBlocks)])

    def forward( self, tX: Tensor, mTime: Tensor ) -> Tensor:
        for oBlock in self.lBlocks:
            tX = oBlock(tX, mTime) #<! Every block receives the same time embedding
        return tX

class AttentionBlock(nn.Module):
    def __init__( self, numCh: int, numHeads: int = 4 ) -> None:
        # Assumes numCh is divisible by numHeads and by 8 for GroupNorm.
        super().__init__()
        self.numHeads = numHeads
        self.oNorm = nn.GroupNorm(8, numCh)
        self.oQKV = nn.Conv2d(numCh, 3 * numCh, 1)
        self.oOut = nn.Conv2d(numCh, numCh, 1)
        nn.init.zeros_(self.oOut.weight) #<! Block starts as the identity
        nn.init.zeros_(self.oOut.bias)

    def forward( self, tX: Tensor ) -> Tensor:
        numB, numCh, numRows, numCols = tX.shape
        tQ, tK, tV = self.oQKV(self.oNorm(tX)).view(numB, 3, self.numHeads, numCh // self.numHeads, numRows * numCols).transpose(-1, -2).unbind(1) #<! Each: B x Heads x (H W) x (C / Heads)
        tZ = F.scaled_dot_product_attention(tQ, tK, tV) #<! Every position attends to all positions
        return tX + self.oOut(tZ.transpose(-1, -2).reshape(numB, numCh, numRows, numCols))

class ConditionalUNet(nn.Module):
    def __init__( self, baseCh: int = 32, timeDim: int = 128, numBlocks: int = 2, *, lChMult: Tuple[int, int, int, int] = (1, 2, 4, 8), useSeparable: bool = False ) -> None:
        # Assumes timeDim is even and >= 4, and numBlocks >= 1.
        # lChMult: channel multiplier per level (full resolution -> 1/8); every baseCh * mult must be a multiple of 8 (GroupNorm) and the last of 4 (attention heads).
        super().__init__()
        ch1, ch2, ch3, ch4 = (baseCh * chMult for chMult in lChMult)
        self.vFreq = torch.exp(-np.log(10000.0) * torch.arange(timeDim // 2) / (timeDim // 2 - 1))
        self.oTime = nn.Sequential(nn.Linear(timeDim, timeDim), nn.SiLU(), nn.Linear(timeDim, timeDim))
        self.oEnc1 = TimeStage(7, ch1, timeDim, numBlocks, useSeparable = useSeparable)
        self.oEnc2 = TimeStage(ch1, ch2, timeDim, numBlocks, useSeparable = useSeparable)
        self.oEnc3 = TimeStage(ch2, ch3, timeDim, numBlocks, useSeparable = useSeparable)
        self.oEnc4 = TimeStage(ch3, ch4, timeDim, numBlocks, useSeparable = useSeparable)
        self.oMid = TimeBlock(ch4, ch4, timeDim, useSeparable = useSeparable)
        self.oAttn = AttentionBlock(ch4) #<! Global mixing at the 16 x 16 bottleneck
        self.oUpsample = nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False)
        self.oDec4 = TimeStage(2 * ch4, ch4, timeDim, numBlocks, useSeparable = useSeparable)
        self.oDec3 = TimeStage(ch4 + ch3, ch3, timeDim, numBlocks, useSeparable = useSeparable)
        self.oDec2 = TimeStage(ch3 + ch2, ch2, timeDim, numBlocks, useSeparable = useSeparable)
        self.oDec1 = TimeStage(ch2 + ch1, ch1, timeDim, numBlocks, useSeparable = useSeparable)
        self.oOut = nn.Conv2d(ch1, 3, 1)

    def forward( self, tNoisy: Tensor, vTime: Tensor, tSource: Tensor, vCondition: Tensor ) -> Tensor:
        # Assumes aligned B x 3 x H x W images, H/W divisible by 16, and B-element time/condition vectors.
        # vCondition is 1 for a present aerial image and 0 for the null condition.
        self.vFreq = self.vFreq.to(tNoisy.device)
        mAngles = vTime.float()[:, None] * self.vFreq[None, :]
        mTime = self.oTime(torch.cat((mAngles.sin(), mAngles.cos()), dim = 1))
        tPresent = vCondition.to(tNoisy.dtype).view(-1, 1, 1, 1)
        tMask = tPresent.expand(-1, 1, *tNoisy.shape[-2:])
        tInput = torch.cat((tNoisy, tSource * tPresent, tMask), dim = 1)
        tEnc1 = self.oEnc1(tInput, mTime)
        tEnc2 = self.oEnc2(F.avg_pool2d(tEnc1, 2), mTime)
        tEnc3 = self.oEnc3(F.avg_pool2d(tEnc2, 2), mTime)
        tEnc4 = self.oEnc4(F.avg_pool2d(tEnc3, 2), mTime)
        tZ = self.oAttn(self.oMid(F.avg_pool2d(tEnc4, 2), mTime))
        for tSkip, oBlock in [(tEnc4, self.oDec4), (tEnc3, self.oDec3), (tEnc2, self.oDec2), (tEnc1, self.oDec1)]:
            tZ = self.oUpsample(tZ)
            tZ = oBlock(torch.cat((tZ, tSkip), dim = 1), mTime)
        return self.oOut(tZ)



In [ ]:
# Model

oModel = ConditionalUNet(baseCh, numBlocks = numBlocks, lChMult = lChMult, useSeparable = useSeparable)
print(f'Model parameters: {sum(tParam.numel() for tParam in oModel.parameters()) / 1e6:.2f} [M]')

# Run device
runDevice = torch.device('cuda:0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'Running on device: {runDevice}')

In [ ]:
# Model Summary

tuInputSize = [tY.shape, (tY.shape[0],), tX.shape, (tX.shape[0],)]
torchinfo.summary(oModel, tuInputSize, col_names = ['kernel_size', 'input_size', 'output_size', 'num_params'], device = runDevice, row_settings = ['depth', 'var_names'])

In [ ]:
# Model Input / Output

oModel = oModel.to(runDevice).eval()
with torch.inference_mode():
    tTest = torch.randn(1, 3, 32, 32, device = runDevice)
    tSource = torch.randn_like(tTest)
    vTestTime = torch.tensor([numDiffSteps // 2], device = runDevice)
    vPresent = torch.ones(1, device = runDevice)
    tPred = oModel(tTest, vTestTime, tSource, vPresent)
    tNull = oModel(tTest, vTestTime, tSource, torch.zeros_like(vPresent))
    tNullOther = oModel(tTest, vTestTime, tSource + 1, torch.zeros_like(vPresent))
print(f'Network output ({predictType}): {tPred.shape}')
print(f'Dropped source is ignored: {torch.equal(tNull, tNullOther)}')

## Train the Model

**Learn to recover the map from its noisy version, using the aerial image as a clue.**

For each training batch:

1. Scale aerial images to $[-1,1]$ and maps to the normalized diffusion space.
2. Pick a random timestep for each map. Add Gaussian noise.
3. Hide some aerial images. This trains the unconditional branch used by CFG.
4. Predict the target given by `predictType`: the added noise ($\boldsymbol{\epsilon}$) or the clean map ($\boldsymbol{y}_0$). Compare with MSE.
5. Backpropagate and update the model.

Each training example uses one sampled timestep, not the full reverse chain. Random timesteps teach the model to handle different noise levels.

### Predict the Noise or Predict the Map?

**Same optimal solution, very different training signal.**

Given $\boldsymbol{y}_t = \sqrt{\bar\alpha_t} \boldsymbol{y}_0 + \sqrt{1 - \bar\alpha_t} \boldsymbol{\epsilon}$, knowing $\boldsymbol{\epsilon}$ is knowing $\boldsymbol{y}_0$ and vice versa. A perfectly trained network is identical under both choices. Yet the MSE on one is a *weighted* MSE on the other:

$$ {\left\| \boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}} \right\|}^{2} = \underbrace{\frac{\bar\alpha_t}{1 - \bar\alpha_t}}_{\text{SNR}(t)} {\left\| \boldsymbol{y}_0 - \hat{\boldsymbol{y}}_0 \right\|}^{2} $$

With the cosine schedule $\text{SNR}(t)$ spans $10^{4}$ (almost clean) to $10^{-4}$ (almost pure noise).

 - **Noise prediction** (`'Noise'`, the DDPM default) weights the low noise steps heavily: fine detail, "clean up an almost clean map". At high noise the target is essentially $\boldsymbol{y}_t$ itself; the whole map enters the target scaled by $\sqrt{\bar\alpha_t} \approx 0.01$. Ignoring the aerial image there costs almost nothing in the loss.
 - **Clean prediction** (`'Clean'`) weights the high noise steps heavily. At $t \approx T$ the noisy map carries no information, so the network is trained as a plain aerial $\to$ map regressor, with full weight, from the first iteration. Low noise steps are trivial ($\boldsymbol{y}_t \approx \boldsymbol{y}_0$) and contribute little.

For a conditional task where the condition matters most when the noise is high, clean prediction learns the conditioning orders of magnitude faster. It also spares the sampler the $1 / \sqrt{\bar\alpha_t}$ division when recovering $\hat{\boldsymbol{y}}_0$ from $\hat{\boldsymbol{\epsilon}}$, which amplifies output errors by $\approx 100$ at the first reverse steps.

* <font color='brown'>(**#**)</font> This choice is called the *prediction parameterization*. A third option, $\boldsymbol{v}$ prediction, mixes both. Loss weighting schemes (P2, Min-SNR-$\gamma$) reach a similar effect while keeping the noise output, yet they cannot fix the $1 / \sqrt{\bar\alpha_t}$ amplification.
* <font color='red'>(**?**)</font> For unconditional generation of natural images, noise prediction is usually preferred. Why?

### Training Budget

**Diffusion denoisers need many iterations, at every noise level.**

Each iteration shows each map at a single random noise level out of `numDiffSteps = 200`.  
One epoch over ~3300 pairs covers each pair once, at one level only.

The `numEpochs = 250` budget gives ~100k iterations at `batchSize = 8`.  
Fewer epochs produce blurry, low contrast maps even when the training loss looks flat.

The learning rate follows a *warmup - hold - cosine* schedule (`WarmupHoldCosine()` with `LambdaLR`):

1. **Warmup** (`warmupFrac = 0.05`): linear ramp to the peak `ηOpt = 2e-4`. Early gradients are large (most steps are clipped) and the attention block is still near identity; a small learning rate avoids a destructive first few epochs.
2. **Hold** (`holdFrac = 0.45`): constant at the peak. The denoiser has to cover $200$ noise levels $\times$ ~3300 pairs; most of the budget is spent here, at a learning rate that still makes progress.
3. **Cosine decay** (remaining 50%): smooth decay to the floor `ηMin = 1e-5`. The decay averages out the noise of the stochastic gradient; the floor keeps the last epochs useful.

* <font color='brown'>(**#**)</font> A _one cycle_ schedule (warmup then a cosine decay to $\approx 10^{-6}$) spends only ~45% of the epochs above $10^{-4}$ and freezes the last quarter; the loss plateau it produces is a scheduler artifact, not convergence.
* <font color='brown'>(**#**)</font> With constant learning rate plus an exponential moving average (EMA) of the weights, the decay phase can be shortened. EMA is the standard choice for diffusion models.

### Loss, Diagnostic and Task Score

**Training computes the loss. Validation computes the loss, a conditioning diagnostic, and the generated map score.**

Every epoch, on the `valNumSamples = 32` held-out pairs:

1. **Validation loss** - the same computation as a training step (random timestep, added noise, dropped conditions) without the update. A fixed seed keeps the noise identical across epochs.
2. **Wrong aerial loss** - the same computation with each map paired to *another* image's aerial. If the network ignores the condition the two losses are equal. The gap is the single most informative number for a conditional model: it shows the conditioning being learned (or not) long before the generated maps do.

Every `valEvery = 10` epochs, generate maps for all held-out aerial images:

1. Start from Gaussian noise, not a corrupted reference map.
2. Generate the map through all reverse steps, conditioned only on the aerial image.
3. Compare the final map with its reference in $[0,1]$, using `Pix2PixScore` with `scoreType = 'R2'`.

Compute one $R^2$ over the entire 32-map validation set. Keep its order, batch size, sampling seed, and guidance scale fixed. Save local `BestModel.pt` only when a newly computed score improves.

* <font color='brown'>(**#**)</font> $R^2$ is a harsh score for maps: a flat image of the mean color scores $\approx 0.37$, the reference shifted by 3 pixels scores $\approx 0.6$. Judge progress by the generated maps together with the score.
* <font color='brown'>(**#**)</font> A low loss does not establish accurate maps. The map score measures the end task; the loss and the wrong aerial gap diagnose the denoiser. Reference maps enter the score and the loss, never the sampler.

### Automatic Mixed Precision with Autocast

**Use lower precision where it helps. Keep higher precision where it matters.**

Automatic Mixed Precision (AMP) can save GPU memory and utilize the memory throughput more efficiently.  
This enables a faster training.

Operations within `torch.autocast` context are dispatched by precision for each operation:
 * Eligible CUDA operations use `Float16`.  
 * Numerically sensitive operations stay in `Float32`.
 * Model weights remain `Float32`.

For instance, in the forward pass:

 * Stored model weights remain `Float32`.
 * `nn.Conv2d` and `nn.Linear` use `Float16` inputs and temporary `Float16` copies of weights. Their outputs are `Float16`.
 * `nn.GroupNorm` runs in `Float32`.
 * Other operations follow their autocast rules and input types.
 * The final convolution outputs `Float16`. `tPred.float()` converts it to `Float32` for the loss and reverse updates.
 * Diffusion updates stay in `Float32`.

#### Vanishing Gradients and AMP

`Float16` can round tiny gradients to zero.  
PyTorch's `GradScaler` temporarily enlarges the loss, which also enlarges the gradients.

1. Scale the loss and backpropagate.
2. Unscale the gradients. Then clip their norm.
3. Update the weights. Skip the update if gradients contain infinity or NaN.
4. Adjust the scale for the next iteration.

Scaling protects small gradients.

* <font color='brown'>(**#**)</font> Validation and sampling need no gradient scaling. They do not backpropagate. 
* <font color='brown'>(**#**)</font> AMP changes numerical precision. Yet it does not change the objectives.


In [ ]:
# Loss and Score

def SSIMScore( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the Structural Similarity Index Measure (SSIM) between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return structural_similarity_index_measure(tYHat, tY, data_range = 1.0)

def ImageR2Score( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the R2 score between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return r2_score(tYHat.flatten(), tY.flatten(), multioutput = 'uniform_average')

class Pix2PixScore(nn.Module):
    def __init__( self, scoreType: Literal['SSIM', 'R2'] = 'SSIM' ) -> None:
        """
        Image quality score for image to image regression.

        Parameters
        ----------
        scoreType : Literal['SSIM', 'R2'], optional
            Score function to use, by default 'SSIM'.
        """
        super().__init__()

        match scoreType:
            case 'SSIM':
                self.hScore = SSIMScore
            case 'R2':
                self.hScore = ImageR2Score
            case _:
                raise ValueError('The parameter `scoreType` must be either `SSIM` or `R2`')

    def forward( self, tYHat: Tensor, tY: Tensor ) -> Tensor:
        """
        Computes the selected score between the generated and target images.

        Parameters
        ----------
        tYHat : Tensor
            Generated image tensor (B x C x H x W).
        tY : Tensor
            Target image tensor (B x C x H x W).

        Returns
        -------
        Tensor
            Scalar image quality score.
        """

        return self.hScore(tYHat, tY)

In [ ]:
# Loss and Score

hL = nn.MSELoss()
hS = Pix2PixScore(scoreType = scoreType)
hL = hL.to(runDevice)
hS = hS.to(runDevice)

In [ ]:
# Training / Validation Epoch

def RunDiffusionEpoch( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hL: Callable, oOpt, *, oScaler = None, dropProb: float = 0.1 ) -> float:
    # Assumes a nonempty training loader of aligned RGB pairs in [0, 1].
    epochLoss = 0.0
    numSamples = 0
    numBatches = len(dlData)
    runDevice = next(oModel.parameters()).device
    oModel.train(True)

    for batchIdx, (tX, tY) in enumerate(dlData):
        # With pinned CPU memory, `non_blocking` lets the CPU schedule work while data transfers to CUDA.
        # Less CPU waiting can reduce runtime; operations in the same CUDA stream still wait for the data.
        tX = tX.to(runDevice, non_blocking = True) * 2 - 1 #<! Aerial condition: [0, 1] -> [-1, 1]
        tY = oDiff.Normalize(tY.to(runDevice, non_blocking = True)) #<! Clean target map: [0, 1] -> diffusion space
        batchSize = tY.shape[0]
        vTime = torch.randint(oDiff.numSteps, (batchSize,), device = runDevice) #<! One timestep per map
        tNoise = torch.randn(tY.shape, device = runDevice)
        tNoisy = oDiff.AddNoise(tY, vTime, tNoise)
        vCondition = (torch.rand(batchSize, device = runDevice) >= dropProb).float()

        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tPred = oModel(tNoisy, vTime, tX, vCondition)
        valLoss = hL(tPred.float(), oDiff.Target(tY, tNoise)) #<! Noise or clean map, per `predictType`
        oOpt.zero_grad()
        if oScaler is not None:
            oScaler.scale(valLoss).backward() #<! Protect small gradients from Float16 underflow
            oScaler.unscale_(oOpt) #<! Unscale before clipping
            nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
            oScaler.step(oOpt)
            oScaler.update()
        else:
            valLoss.backward()
            nn.utils.clip_grad_norm_(oModel.parameters(), 1.0)
            oOpt.step()

        epochLoss += batchSize * valLoss.detach().item()
        numSamples += batchSize
        print(f'\rTrain Loss - Iteration: {(batchIdx + 1):3d} / {numBatches}, Loss: {valLoss:.6f}', end = '')

    print('\r' + ' ' * 80, end = '\r') #<! Blank the progress line so a shorter line does not leave tail characters
    return epochLoss / numSamples

@torch.inference_mode()
def EvaluateLoss( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hL: Callable, *, dropProb: float = 0.1, sampleSeed: int = 512 ) -> Tuple[float, float]:
    # Same computation as a training step without the update; the fixed seed keeps the noise identical across epochs.
    # Also the loss with a wrong (shuffled) aerial image: the gap to the true loss measures how much the network uses the condition.
    runDevice = next(oModel.parameters()).device
    oGen = torch.Generator(device = runDevice).manual_seed(sampleSeed)
    oModel.eval()
    epochLoss = 0.0
    epochLossShuffled = 0.0
    numSamples = 0

    for tX, tY in dlData:
        tX = tX.to(runDevice, non_blocking = True) * 2 - 1
        tY = oDiff.Normalize(tY.to(runDevice, non_blocking = True))
        batchSize = tY.shape[0]
        vTime = torch.randint(oDiff.numSteps, (batchSize,), device = runDevice, generator = oGen)
        tNoise = torch.randn(tY.shape, device = runDevice, generator = oGen)
        tNoisy = oDiff.AddNoise(tY, vTime, tNoise)
        vCondition = (torch.rand(batchSize, device = runDevice, generator = oGen) >= dropProb).float()
        tTarget = oDiff.Target(tY, tNoise)

        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tPred = oModel(tNoisy, vTime, tX, vCondition)
            tPredShuffled = oModel(tNoisy, vTime, tX.roll(1, dims = 0), vCondition) #<! Each map paired with another image's aerial
        epochLoss += batchSize * hL(tPred.float(), tTarget).item()
        epochLossShuffled += batchSize * hL(tPredShuffled.float(), tTarget).item()
        numSamples += batchSize

    return epochLoss / numSamples, epochLossShuffled / numSamples

@torch.inference_mode()
def EvaluateDiffusionModel( oModel: nn.Module, oDiff: DiffusionSchedule, dlData, hS: Callable, *, guidanceScale: float = 2.0, sampleSeed: int = 512 ) -> float:
    # Assumes a nonempty, fixed-order loader with images in [0, 1].
    runDevice = next(oModel.parameters()).device
    lGenerated, lTarget = [], []
    for batchIdx, (tX, tY) in enumerate(dlData):
        tGenerated, _, _ = SampleMaps(oModel, oDiff, tX, runDevice, guidanceScale = guidanceScale, sampleSeed = sampleSeed + batchIdx, numFrames = 0) #<! Generate from noise and aerial images only
        lGenerated.append(tGenerated)
        lTarget.append(tY.cpu()) #<! Reference maps are used only by the score
        print(f'\rVal Map Score - Batch: {batchIdx + 1:3d} / {len(dlData)}', end = '')
    print('\r' + ' ' * 80, end = '\r')
    return hS(torch.cat(lGenerated), torch.cat(lTarget)).item() #<! One score over all supplied validation maps

def TrainDiffusionModel( oModel: nn.Module, oDiff: DiffusionSchedule, dlTrain, dlVal, oOpt, numEpoch: int, hL: Callable, hS: Callable, *, oSch = None, oScaler = None, dropProb: float = 0.1, guidanceScale: float = 2.0, valEvery: int = 5, sampleSeed: int = 512 ) -> Tuple[nn.Module, List[float], List[float], List[float], List[int], List[float], List[float]]:
    # Assumes nonempty loaders and numEpoch > 0. Only scheduled epochs generate validation maps.
    if valEvery < 1:
        raise ValueError('valEvery must be positive')
    lTrainLoss, lValLoss, lValLossShuffled, lLearnRate = [], [], [], []
    lValEpoch, lValScore = [], []
    bestScore = -float('inf')
    totalStartTime = time.time()

    for epochIdx in range(numEpoch):
        startTime = time.time()
        learnRate = oOpt.param_groups[0]['lr']
        trainLoss = RunDiffusionEpoch(oModel, oDiff, dlTrain, hL, oOpt, oScaler = oScaler, dropProb = dropProb)
        valLoss, valLossShuffled = EvaluateLoss(oModel, oDiff, dlVal, hL, dropProb = dropProb, sampleSeed = sampleSeed) #<! Cheap: two forward passes per validation batch
        scoreEpoch = (epochIdx + 1) % valEvery == 0
        if scoreEpoch:
            valScr = EvaluateDiffusionModel(oModel, oDiff, dlVal, hS, guidanceScale = guidanceScale, sampleSeed = sampleSeed)
            lValEpoch.append(epochIdx + 1)
            lValScore.append(valScr)
        if oSch is not None:
            oSch.step()
        epochTime = time.time() - startTime

        lTrainLoss.append(trainLoss)
        lValLoss.append(valLoss)
        lValLossShuffled.append(valLossShuffled)
        lLearnRate.append(learnRate)
        print(f'Epoch {(epochIdx + 1):4d} / {numEpoch}', end = '')
        print(f' | Train Loss: {trainLoss:7.5f}', end = '')
        print(f' | Val Loss: {valLoss:7.5f} (Wrong Aerial: {valLossShuffled:7.5f})', end = '')
        if scoreEpoch:
            print(f' | Val Score: {valScr:6.3f}', end = '')
        print(f' | Epoch Time: {epochTime:5.2f}', end = '')

        if scoreEpoch and valScr > bestScore: #<! Never checkpoint on an unscored epoch
            bestScore = valScr
            try:
                dCheckPoint = {'Model': oModel.state_dict(), 'Optimizer': oOpt.state_dict(), 'MapMean': oDiff.tMean.flatten().tolist(), 'MapStd': oDiff.dataStd, 'PredictType': oDiff.predictType}
                if oSch is not None:
                    dCheckPoint['Scheduler'] = oSch.state_dict()
                torch.save(dCheckPoint, 'BestModel.pt')
                print(' | <-- Checkpoint!', end = '')
            except OSError as oError:
                print(f' | <-- Failed: {oError}', end = '')
        print(' |')

    totalTime = time.time() - totalStartTime
    print(f'Total Training Time: {totalTime:.2f} [Sec] ({time.strftime("%H:%M:%S", time.gmtime(totalTime))})')

    return oModel, lTrainLoss, lValLoss, lValLossShuffled, lValEpoch, lValScore, lLearnRate

### Generate Maps with CFG

Initialize a target-shaped Gaussian tensor and keep the aerial image fixed. At every step, run the network with and without the condition, combine the two outputs using `guidanceScale`, recover the clean map estimate with `PredictClean()`, and apply the DDPM posterior update.

`SampleMaps()` takes only aerial images, not target maps. It returns generated RGB maps in $[0,1]$ and display snapshots for the first image. Ground-truth maps are used only for evaluation.

* <font color='brown'>(**#**)</font> The CFG combination is linear, so it applies equally to a noise output and to a clean map output.
* <font color='red'>(**?**)</font> Why would CFG fail if the model were never trained with dropped conditions?
* <font color='brown'>(**#**)</font> This is a full DDPM chain. Do not skip timesteps to accelerate sampling without changing to an appropriate sampler such as DDIM.


In [ ]:
# Classifier-Free Guided DDPM Sampling

def PredictGuided( oModel: nn.Module, tNoisy: Tensor, vTime: Tensor, tSource: Tensor, guidanceScale: float ) -> Tensor:
    # Classifier free guidance on the network output (valid for both noise and clean prediction, the combination is linear)
    vPresent = torch.ones(len(tNoisy), device = tNoisy.device) #<! One condition-presence flag per image
    if guidanceScale == 1: #<! Ordinary conditional prediction needs only one model call
        return oModel(tNoisy, vTime, tSource, vPresent).float()
    tPredNull = oModel(tNoisy, vTime, tSource, torch.zeros_like(vPresent)).float() #<! Zero mask hides the aerial image
    if guidanceScale == 0:
        return tPredNull
    tPredCond = oModel(tNoisy, vTime, tSource, vPresent).float()
    return tPredNull + guidanceScale * (tPredCond - tPredNull) #<! Amplify the condition's effect on the output

@torch.inference_mode()
def SampleMaps( oModel: nn.Module, oDiff: DiffusionSchedule, tSource: Tensor, runDevice: torch.device, *, guidanceScale: float = 2.0, sampleSeed: int = 512, numFrames: int = 6 ) -> Tuple[Tensor, list, list]:
    """Generate maps from BCHW aerial images in [0, 1]; numFrames = 0 disables snapshots."""
    oModel.eval()
    tSource = tSource.to(runDevice, non_blocking = True) * 2 - 1 #<! Same condition normalization as training
    oGen = torch.Generator(device = runDevice).manual_seed(sampleSeed)
    tNoisy = torch.randn(tSource.shape, device = runDevice, generator = oGen) #<! No target image is supplied
    lFrames, lSteps = [], []
    if numFrames > 0:
        lFrames.append(oDiff.Denormalize(tNoisy[:1]).cpu())
        lSteps.append(oDiff.numSteps)
    vSaveSteps = np.linspace(oDiff.numSteps, 0, numFrames, dtype = int)
    for stepIdx in reversed(range(oDiff.numSteps)):
        vTime = torch.full((len(tSource),), stepIdx, device = runDevice, dtype = torch.long)
        with torch.autocast(device_type = runDevice.type, enabled = runDevice.type == 'cuda'):
            tPred = PredictGuided(oModel, tNoisy, vTime, tSource, guidanceScale) #<! Same aerial image at every step
        tNoise = torch.randn(tNoisy.shape, device = runDevice, generator = oGen) if stepIdx > 0 else torch.zeros_like(tNoisy)
        tNoisy = oDiff.Step(tNoisy, tPred, stepIdx, tNoise) #<! Reverse update stays in Float32
        if stepIdx in vSaveSteps:
            lFrames.append(oDiff.Denormalize(tNoisy[:1]).cpu())
            lSteps.append(stepIdx)
    return oDiff.Denormalize(tNoisy).cpu(), lFrames, lSteps

In [ ]:
# Optimizer Related

def WarmupHoldCosine( numEpochs: int, warmupFrac: float, holdFrac: float, minRatio: float ) -> Callable[[int], float]:
    # LR multiplier per epoch: linear warmup -> hold at the peak -> cosine decay to `minRatio` of the peak (never reaches zero)
    numWarmup = max(1, round(warmupFrac * numEpochs))
    numHold   = round(holdFrac * numEpochs)
    numDecay  = max(1, numEpochs - numWarmup - numHold)

    def hLrRatio( epochIdx: int ) -> float:
        if epochIdx < numWarmup:
            return (epochIdx + 1) / numWarmup
        if epochIdx < numWarmup + numHold:
            return 1.0
        decayFrac = min(1.0, (epochIdx - numWarmup - numHold) / numDecay)
        return minRatio + (1.0 - minRatio) * 0.5 * (1.0 + np.cos(np.pi * decayFrac))

    return hLrRatio

oDiff = DiffusionSchedule(numDiffSteps, runDevice, vMean = vMapMean, dataStd = mapStd, predictType = predictType) #<! Same statistics, on the run device
oOpt = torch.optim.AdamW(oModel.parameters(), lr = ηOpt, betas = tuβ, weight_decay = weightDecay)
oSch = torch.optim.lr_scheduler.LambdaLR(oOpt, WarmupHoldCosine(numEpochs, warmupFrac, holdFrac, ηMin / ηOpt)) #<! Warm up over 5%, hold at 2e-4 until 50%, cosine to 1e-5
oScaler = torch.amp.GradScaler('cuda', enabled = runDevice.type == 'cuda')



In [ ]:
# Training Loop

oModel = oModel.to(runDevice)
_, lTrainLoss, lValLoss, lValLossShuffled, lValEpoch, lValScore, lLearnRate = TrainDiffusionModel(oModel, oDiff, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch, oScaler = oScaler, dropProb = conditionDropProb, guidanceScale = guidanceScale, valEvery = valEvery, sampleSeed = seedNum)

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (18, 5))
vHa = np.ravel(vHa)
vEpoch = np.arange(1, len(lTrainLoss) + 1)

hA = vHa[0]
hA.plot(vEpoch, lTrainLoss, lw = 2, label = 'Train')
hA.plot(vEpoch, lValLoss, lw = 2, label = 'Validation')
hA.plot(vEpoch, lValLossShuffled, lw = 2, ls = '--', label = 'Validation (Wrong Aerial)')
hA.set_title(f'Loss ({predictType} Prediction)')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lValEpoch, lValScore, 'o-', lw = 2, label = 'Validation')
hA.set_title(f'Generated Map Score ({len(dlVal.dataset)} Maps)')
hA.set_xlabel('Epoch')
hA.set_ylabel(scoreType)
hA.legend()

hA = vHa[2]
hA.plot(vEpoch, lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

In [ ]:
# Load the Model

dModel = torch.load('BestModel.pt', map_location = runDevice, weights_only = True)
oModel.load_state_dict(dModel['Model']) #<! Requires the same `baseCh`, `lChMult`, `numBlocks` as the trained model
print(f'Model loaded from: BestModel.pt ({"EMA" if "ModelRaw" in dModel else "live"} weights)')

# The schedule must match the one the weights were trained with (older checkpoints fall back to the values computed above)
oDiff = DiffusionSchedule(numDiffSteps, runDevice, vMean = tuple(dModel.get('MapMean', vMapMean)), dataStd = dModel.get('MapStd', mapStd), predictType = dModel.get('PredictType', predictType))
print(f'Schedule: {oDiff.predictType} prediction, map mean {[round(v, 3) for v in oDiff.tMean.flatten().tolist()]}, std {oDiff.dataStd:.3f}')
oModel.eval();

In [ ]:
# Compute the Score on the Validation Set

valScore = EvaluateDiffusionModel(oModel, oDiff, dlVal, hS, guidanceScale = 2, sampleSeed = seedNum)
print(f'Validation Score ({len(dlVal.dataset)} Maps): {valScore:.4f}')

In [ ]:
# Aerial Image -> Conditional DDPM -> Map

lPairs = [dsVal[sampleIdx] for sampleIdx in range(min(numImg, len(dsVal)))]
tSource = torch.stack([tuPair[0] for tuPair in lPairs])
tTarget = torch.stack([tuPair[1] for tuPair in lPairs])
tGenerated, lFrames, lSteps = SampleMaps(oModel, oDiff, tSource, runDevice, guidanceScale = guidanceScale, sampleSeed = seedNum, numFrames = numPlotSteps)

hF, mHa = plt.subplots(len(lPairs), 3, figsize = (12, 4 * len(lPairs)), squeeze = False)
for sampleIdx, vHa in enumerate(mHa):
    for hA, tImage, title in zip(vHa, (tSource[sampleIdx], tTarget[sampleIdx], tGenerated[sampleIdx]),
                               ('Aerial Image', 'Target Map', f'Generated Map (CFG {guidanceScale:g})')):
        hA.imshow(TensorImgNumpy(tImage))
        hA.set_title(title)
        hA.axis('off')
hF.tight_layout()

mae = F.l1_loss(tGenerated, tTarget).item()
ssim = structural_similarity_index_measure(tGenerated, tTarget, data_range = 1.0).item()
r2 = hS(tGenerated, tTarget).item()
print(f'Preview only ({len(lPairs)} validation pairs): MAE = {mae:.4f}, SSIM = {ssim:.4f}, R2 = {r2:.4f}')

In [ ]:
# Follow the Reverse Process

tCondition, tReference = dsVal[0]
tMap, lFrames, lSteps = SampleMaps(oModel, oDiff, tCondition[None], runDevice, guidanceScale = guidanceScale, sampleSeed = seedNum, numFrames = numPlotSteps) #<! Rebuild frames for this exact aerial image
numPanels = len(lFrames) + 2
numCols = 4
numRows = (numPanels + numCols - 1) // numCols
hF, mHa = plt.subplots(numRows, numCols, figsize = (4 * numCols, 4 * numRows), squeeze = False)
vHa = mHa.ravel()
vHa[0].imshow(TensorImgNumpy(tCondition))
vHa[0].set_title('Aerial Condition (Fixed)')
vHa[1].imshow(TensorImgNumpy(tReference))
vHa[1].set_title('Reference Map (Not an Input)')
for hA, tFrame, stepIdx in zip(vHa[2:], lFrames, lSteps):
    hA.imshow(TensorImgNumpy(tFrame[0]))
    hA.set_title('Generated Map' if stepIdx == 0 else f'Reverse Steps Remaining: {stepIdx}')
for hA in vHa:
    hA.axis('off')
hF.tight_layout()

In [ ]:
# Compare Guidance with Identical Initial and Per Step Noise
lGuidance = [0.0, 1.0, 2.0]
hF, vHa = plt.subplots(1, len(lGuidance) + 2, figsize = (4 * (len(lGuidance) + 2), 4))
vHa[0].imshow(TensorImgNumpy(tCondition)); vHa[0].set_title('Aerial Condition'); vHa[0].axis('off')
vHa[1].imshow(TensorImgNumpy(tReference)); vHa[1].set_title('Reference Map'); vHa[1].axis('off')
for hA, scale in zip(vHa[2:], lGuidance):
    tMap, _, _ = SampleMaps(oModel, oDiff, tCondition[None], runDevice, guidanceScale = scale, sampleSeed = seedNum, numFrames = 0)
    hA.imshow(TensorImgNumpy(tMap[0]))
    hA.set_title(f'CFG Scale = {scale:g}')
    hA.axis('off')
hF.tight_layout();

In [ ]:
# Compute the Score on the Validation Set per CFG Scale

for valCfg in [0.0, 1.0, 2.0]:
    valScore = EvaluateDiffusionModel(oModel, oDiff, dlVal, hS, guidanceScale = valCfg, sampleSeed = seedNum)
    print(f'Validation Score ({len(dlVal.dataset)} Maps) with CFG Scale = {valCfg:g}: {valScore:.4f}')

* <font color='blue'>(**!**)</font> Change `guidanceScale` without retraining. Compare 0, 1, 2, and 4 with the same seed and aerial image.
* <font color='red'>(**?**)</font> Does stronger guidance always improve road alignment? Look for color saturation, artifacts, and missing small structures.
* <font color='green'>(**@**)</font> Compare multiple seeds for one aerial image to inspect diversity. Not every generated difference represents calibrated geographic uncertainty.
* <font color='green'>(**@**)</font> Compare generated-map R2 across guidance scales using the same validation pairs and sampling seed. The preview metrics cover only displayed pairs; periodic validation scores cover the entire 32-map validation set.
* <font color='green'>(**@**)</font> Swap or shuffle the aerial conditions with identical sampling noise to check whether the generated maps follow the source. Compare CFG 1 with stronger guidance before attributing artifacts to the architecture.
* <font color='brown'>(**#**)</font> Diffusion is a generative model, not a guarantee of accurate cartography. Do not treat generated map details as verified geographic facts.
